In [0]:
%%capture
!pip install kagglehub[pandas-datasets]

In [0]:
import kagglehub
from kagglehub import KaggleDatasetAdapter

# Set the path to the file you'd like to load
file_path = "movies_metadata.csv"

# Load the latest version
movies = kagglehub.load_dataset(
  KaggleDatasetAdapter.PANDAS,
  "rounakbanik/the-movies-dataset",
  file_path,
  # Provide any additional arguments like 
  # sql_query or pandas_kwargs. See the 
  # documenation for more information:
  # https://github.com/Kaggle/kagglehub/blob/main/README.md#kaggledatasetadapterpandas
)

movies.head()

In [0]:
# Set the path to the file you'd like to load
file_path = "credits.csv"

# Load the latest version
credits = kagglehub.load_dataset(
  KaggleDatasetAdapter.PANDAS,
  "rounakbanik/the-movies-dataset",
  file_path,
)
credits.head()

In [0]:
# Set the path to the file you'd like to load
file_path = "keywords.csv"

# Load the latest version
keywords = kagglehub.load_dataset(
  KaggleDatasetAdapter.PANDAS,
  "rounakbanik/the-movies-dataset",
  file_path,
)
keywords.head()

In [0]:
# Set the path to the file you'd like to load
file_path = "ratings.csv"

# Load the latest version
ratings = kagglehub.load_dataset(
  KaggleDatasetAdapter.PANDAS,
  "rounakbanik/the-movies-dataset",
  file_path,
)
ratings.head()

In [0]:
movie_averages = ratings.groupby('movieId')[['rating']].mean()
movie_averages

In [0]:
import pandas as pd

movies['id'] = pd.to_numeric(movies['id'], errors='coerce')
movies = movies.dropna(subset=['id'])
movies['id'] = movies['id'].astype(int)
df = pd.merge(movies, credits, on='id', how = 'left')
df = pd.merge(df, keywords, on='id', how = 'left')
df = pd.merge(df, movie_averages, left_on='id', right_on='movieId')
df

In [0]:
df.columns.tolist()

In [0]:
df = df[['id', 'title', 'overview', 'keywords', 'vote_average', 'rating', 'original_language', 'release_date', 'runtime', 'genres', 'crew']]
df.head()

In [0]:
df.columns[df.isna().any()].tolist()

In [0]:
df['overview'] = df['overview'].fillna('N/A')
df['original_language'] = df['original_language'].fillna('N/A')
df['release_date'] = df['release_date'].fillna('N/A')
df['runtime'] = df['runtime'].fillna(0)

In [0]:
df.columns.tolist()

In [0]:
df['id'] = df['id'].astype('int64')
df['title'] = df['title'].astype('string')
df['overview'] = df['overview'].astype('string')
df['keywords'] = df['keywords'].astype('string')
df['vote_average'] = df['vote_average'].astype('float')
df['rating'] = df['rating'].astype('float')
df['original_language'] = df['original_language'].astype('string')
df['release_date'] = pd.to_datetime(df['release_date'], errors='coerce')
df['runtime'] = df['runtime'].astype('float')
df['genres'] = df['genres'].astype('string')
df['crew'] = df['crew'].astype('string')

In [0]:
from ast import literal_eval

df['genres'] = df['genres'].apply(literal_eval)
df['keywords'] = df['keywords'].apply(literal_eval)
df['crew'] = df['crew'].apply(literal_eval)

In [0]:
import numpy as np

def get_director(x):
    for i in x:
        if i['job'] == 'Director':
            return i['name']
    return np.nan

In [0]:
df['director'] = df['crew'].apply(get_director)
df = df.drop(columns=['crew'])
df.head()

In [0]:
%pip install nltk

In [0]:
from nltk.stem.snowball import SnowballStemmer
stemmer = SnowballStemmer("english")

In [0]:
test = df.copy()
test.head()

In [0]:
def separate(x):
    result = ''
    for word in x:
        current_word = word['name']
        # current_word = stemmer.stem(current_word)
        result += current_word + ' '
    return result

In [0]:
df['keywords'] = df['keywords'].apply(separate)
df['genres'] = df['genres'].apply(separate)
df.head()

In [0]:
%%capture
pip install sentence-transformers

In [0]:
df.info()

In [0]:
df['genres'] = df['genres'].astype('str')
df['director'] = df['director'].astype('str')
df['keywords'] = df['keywords'].astype('str')
df.info()

In [0]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer('all-miniLm-L6-v2')

texts = df['keywords'].tolist()
embeddings = model.encode(texts)
df['embeddings'] = embeddings.tolist()
# df.head()

## Feature Analysis

In [0]:
print("Rows:", df.shape[0])
print("Columns:", df.shape[1])

In [0]:
df.info()

In [0]:
df.describe()

## Exploratory Data Analysis

In [0]:
df.head()

In [0]:
import matplotlib.pyplot as plt

numerical_categories = ['vote_average', 'rating', 'runtime']
df[numerical_categories].hist(bins=30, figsize=(10, 10), xlabelsize=8, ylabelsize=8)
plt.show()

In [0]:
categorical_categories = ['original_language']

for category in categorical_categories:
    plt.figure(figsize=(10, 5))
    df[category].value_counts().plot(kind='bar')
    plt.title(category)
    plt.xlabel(category)
    plt.ylabel('Count')
    plt.show()

## Cosine Similarlity Recommendation

In [0]:
from sklearn.metrics.pairwise import cosine_similarity

def get_recommendations(title, df, k=5):
    title = title.lower()
    title_embedding = model.encode(title)
    cosine_similarities = cosine_similarity([title_embedding], np.array(df['embeddings'].tolist())).flatten()
    top_k_indices = cosine_similarities.argsort()[-k:][::-1]
    top_k_titles = df.iloc[top_k_indices]['title'].tolist()
    return top_k_titles

get_recommendations('The Godfather', df)